<a href="https://colab.research.google.com/github/Sakti-Cristopel/Dasar-Python/blob/main/Pertemuan%2012_Sakti%20Cristopel%20Lingga_240401010125.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 12 — Asosiasi Data & Sistem Rekomendasi

Nama Lengkap: Sakti Cristopel Lingga

NIM: 240401010125

Kelas: IF403

Topik:
Apriori, Association Rules, dan Content-Based Filtering

Dataset:
Dataset Sintetis Transaksi dan Katalog Produk

## Deskripsi Tugas

Pada praktikum ini dilakukan analisis pola pembelian pelanggan menggunakan Association Rule Mining dengan algoritma Apriori. Data transaksi digunakan untuk menemukan kombinasi produk yang sering dibeli secara bersamaan dan membentuk aturan asosiasi berdasarkan Support, Confidence, dan Lift.

Selain itu, praktikum ini membangun sistem rekomendasi sederhana menggunakan Content-Based Filtering. Pendekatan tersebut menggunakan kesamaan kategori produk dengan Cosine Similarity untuk menemukan produk yang memiliki karakteristik serupa.

Kedua pendekatan kemudian dibandingkan untuk mengetahui perbedaan rekomendasi berdasarkan pola pembelian pelanggan dan kemiripan karakteristik produk.

## Tujuan Praktikum

Praktikum ini bertujuan untuk:

1. Membuat dan mengeksplorasi dataset transaksi sintetis.
2. Mengubah data transaksi menjadi format One-Hot Encoding.
3. Menemukan frequent itemset menggunakan algoritma Apriori.
4. Memahami Support, Confidence, dan Lift.
5. Membentuk dan menganalisis aturan asosiasi.
6. Membuat sistem rekomendasi berbasis Content-Based Filtering.
7. Menggunakan Cosine Similarity untuk mengukur kemiripan produk.
8. Membandingkan Association Rules dengan Content-Based Filtering.
9. Memahami penggunaan pendekatan hybrid dalam sistem rekomendasi.

In [26]:
# Import library utama

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Library untuk Association Rule Mining

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Library untuk Content-Based Filtering

from sklearn.metrics.pairwise import cosine_similarity

print("Semua library berhasil di-import.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Semua library berhasil di-import.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Langkah 1 — Generate dan Eksplorasi Dataset Transaksi

Dataset transaksi dibuat secara sintetis dengan 50 transaksi. Setiap transaksi terdiri dari 2 sampai 5 produk yang dipilih dari 10 jenis produk.

Pada dataset juga disisipkan pola pembelian antara Roti dan Selai. Pola tersebut digunakan untuk mensimulasikan kondisi ketika pelanggan yang membeli Roti memiliki kecenderungan membeli Selai.

In [27]:
# Menentukan seed agar hasil dapat direproduksi

np.random.seed(42)

# Daftar produk

produk = [
    'Roti',
    'Selai',
    'Susu',
    'Sereal',
    'Telur',
    'Keju',
    'Kopi',
    'Gula',
    'Teh',
    'Mentega'
]

# Membuat 50 transaksi

transaksi = []

for _ in range(50):

    n_item = np.random.randint(2, 6)

    transaksi.append(
        list(
            np.random.choice(
                produk,
                n_item,
                replace=False
            )
        )
    )

# Menyisipkan pola Roti -> Selai

for i in range(20):

    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:

        transaksi[i].append('Selai')

print("Contoh transaksi:")
for i, t in enumerate(transaksi[:5], start=1):
    print(f"Transaksi {i}: {t}")

print("\nJumlah transaksi:", len(transaksi))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Contoh transaksi:
Transaksi 1: [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
Transaksi 2: [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
Transaksi 3: [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]
Transaksi 4: [np.str_('Selai'), np.str_('Keju'), np.str_('Telur'), np.str_('Teh')]
Transaksi 5: [np.str_('Mentega'), np.str_('Susu'), np.str_('Gula'), np.str_('Keju')]

Jumlah transaksi: 50


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [28]:
# Menghitung frekuensi setiap produk

frekuensi_produk = {}

for transaksi_satu in transaksi:

    for item in transaksi_satu:

        frekuensi_produk[item] = (
            frekuensi_produk.get(item, 0) + 1
        )

frekuensi_produk = pd.Series(
    frekuensi_produk
).sort_values(
    ascending=False
)

print("Frekuensi setiap produk:")
display(frekuensi_produk)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Frekuensi setiap produk:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,0
Selai,26
Teh,23
Mentega,21
Telur,18
Keju,17
Roti,16
Kopi,16
Susu,16
Gula,16
Sereal,9


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [29]:
# Menghitung frekuensi setiap produk

frekuensi_produk = {}

for transaksi_satu in transaksi:

    for item in transaksi_satu:

        frekuensi_produk[item] = (
            frekuensi_produk.get(item, 0) + 1
        )

frekuensi_produk = pd.Series(
    frekuensi_produk
).sort_values(
    ascending=False
)

print("Frekuensi setiap produk:")
display(frekuensi_produk)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Frekuensi setiap produk:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,0
Selai,26
Teh,23
Mentega,21
Telur,18
Keju,17
Roti,16
Kopi,16
Susu,16
Gula,16
Sereal,9


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Interpretasi Frekuensi Produk

Frekuensi produk menunjukkan berapa banyak transaksi yang mengandung masing-masing produk. Produk dengan frekuensi lebih tinggi merupakan produk yang lebih sering muncul dalam transaksi pelanggan.

Informasi ini memberikan gambaran awal mengenai produk yang populer sebelum dilakukan analisis Association Rule Mining menggunakan algoritma Apriori.

## Langkah 2 — One-Hot Encoding Transaksi

Algoritma Apriori membutuhkan data dalam bentuk biner, yaitu setiap kolom mewakili satu produk dan setiap baris mewakili satu transaksi.

Nilai True menunjukkan bahwa suatu produk terdapat dalam transaksi, sedangkan nilai False menunjukkan produk tersebut tidak terdapat dalam transaksi.

In [30]:
# Membuat TransactionEncoder

te = TransactionEncoder()

# Melakukan encoding

te_ary = te.fit(transaksi).transform(transaksi)

# Mengubah hasil menjadi DataFrame

df_transaksi = pd.DataFrame(
    te_ary,
    columns=te.columns_
)

print("Hasil One-Hot Encoding:")

display(df_transaksi.head())

print("Shape data transaksi:", df_transaksi.shape)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Hasil One-Hot Encoding:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Gula,Keju,Kopi,Mentega,Roti,Selai,Sereal,Susu,Teh,Telur
0,False,True,True,True,True,True,False,False,False,False
1,False,False,True,True,True,True,False,False,True,False
2,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,True,True
4,True,True,False,True,False,False,False,True,False,False


Shape data transaksi: (50, 10)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Interpretasi One-Hot Encoding

Dataset hasil encoding memiliki 50 baris yang mewakili transaksi dan 10 kolom yang mewakili produk. Format ini memungkinkan algoritma Apriori menghitung seberapa sering suatu produk atau kombinasi produk muncul dalam transaksi.

## Langkah 3 — Frequent Itemset dengan Apriori

Algoritma Apriori digunakan untuk menemukan kombinasi produk yang sering muncul bersama dalam transaksi.

Parameter utama yang digunakan adalah min_support. Support menunjukkan proporsi transaksi yang mengandung suatu item atau kombinasi item.

Pada praktikum ini digunakan beberapa nilai min_support, yaitu 0.05, 0.10, dan 0.20 untuk melihat bagaimana perubahan threshold memengaruhi jumlah frequent itemset yang ditemukan.

In [31]:
# Menguji beberapa nilai minimum support

for ms in [0.05, 0.10, 0.20]:

    freq = apriori(
        df_transaksi,
        min_support=ms,
        use_colnames=True
    )

    print(
        f"min_support={ms}: "
        f"{len(freq)} itemset ditemukan"
    )

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [32]:
# Menggunakan min_support = 0.10

freq_items = apriori(
    df_transaksi,
    min_support=0.10,
    use_colnames=True
)

# Mengurutkan berdasarkan support tertinggi

freq_items = freq_items.sort_values(
    'support',
    ascending=False
)

print("10 Frequent Itemset Teratas:")

display(
    freq_items.head(10)
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

10 Frequent Itemset Teratas:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
5,0.52,(Selai)
8,0.46,(Teh)
3,0.42,(Mentega)
9,0.36,(Telur)
1,0.34,(Keju)
0,0.32,(Gula)
2,0.32,(Kopi)
4,0.32,(Roti)
7,0.32,(Susu)
36,0.24,"(Teh, Selai)"


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Interpretasi Frequent Itemset

Nilai min_support menentukan seberapa sering suatu itemset harus muncul agar dianggap frequent. Semakin tinggi nilai min_support, semakin sedikit itemset yang memenuhi syarat.

Pada praktikum ini digunakan min_support sebesar 0.10 karena menghasilkan jumlah itemset yang masih dapat dianalisis dengan baik dan tidak terlalu sedikit maupun terlalu banyak.

## Langkah 4 — Membentuk Aturan Asosiasi

Frequent itemset yang telah ditemukan kemudian digunakan untuk membentuk Association Rules.

Tiga ukuran utama yang digunakan adalah:

- Support: menunjukkan seberapa sering kombinasi item muncul dalam seluruh transaksi.
- Confidence: menunjukkan seberapa besar kemungkinan consequent dibeli ketika antecedent dibeli.
- Lift: menunjukkan kekuatan hubungan antara antecedent dan consequent dibandingkan dengan kondisi jika keduanya muncul secara independen.

Aturan dengan Lift lebih dari 1 menunjukkan adanya hubungan positif antara antecedent dan consequent.

In [33]:
# Membentuk association rules

rules = association_rules(
    freq_items,
    metric='confidence',
    min_threshold=0.5
)

# Memilih aturan dengan lift > 1

rules = rules[
    rules['lift'] > 1
].sort_values(
    'lift',
    ascending=False
)

print("Jumlah aturan dengan Lift > 1:", len(rules))

display(
    rules[
        [
            'antecedents',
            'consequents',
            'support',
            'confidence',
            'lift'
        ]
    ].head(10)
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Jumlah aturan dengan Lift > 1: 16


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
8,"(Keju, Teh)",(Telur),0.12,0.857143,2.380952
14,"(Mentega, Selai)",(Kopi),0.10,0.625000,1.953125
12,"(Gula, Roti)",(Selai),0.10,1.000000,1.923077
7,(Sereal),(Mentega),0.14,0.777778,1.851852
10,"(Teh, Telur)",(Keju),0.12,0.600000,1.764706
15,"(Kopi, Selai)",(Mentega),0.10,0.714286,1.700680
9,"(Keju, Telur)",(Teh),0.12,0.750000,1.630435
11,"(Gula, Selai)",(Roti),0.10,0.500000,1.562500
13,"(Mentega, Kopi)",(Selai),0.10,0.714286,1.373626
1,(Roti),(Selai),0.22,0.687500,1.322115


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [34]:
rules_display = rules[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
].head(10).copy()

rules_display['support'] = (
    rules_display['support'] * 100
).round(2)

rules_display['confidence'] = (
    rules_display['confidence'] * 100
).round(2)

rules_display['lift'] = (
    rules_display['lift']
).round(3)

display(rules_display)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
8,"(Keju, Teh)",(Telur),12.0,85.71,2.381
14,"(Mentega, Selai)",(Kopi),10.0,62.50,1.953
12,"(Gula, Roti)",(Selai),10.0,100.00,1.923
7,(Sereal),(Mentega),14.0,77.78,1.852
10,"(Teh, Telur)",(Keju),12.0,60.00,1.765
15,"(Kopi, Selai)",(Mentega),10.0,71.43,1.701
9,"(Keju, Telur)",(Teh),12.0,75.00,1.630
11,"(Gula, Selai)",(Roti),10.0,50.00,1.562
13,"(Mentega, Kopi)",(Selai),10.0,71.43,1.374
1,(Roti),(Selai),22.0,68.75,1.322


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Interpretasi Association Rules

Aturan asosiasi diurutkan berdasarkan nilai Lift tertinggi. Aturan dengan Lift lebih dari 1 menunjukkan bahwa kemunculan antecedent memiliki hubungan positif dengan consequent.

Aturan dengan Lift tertinggi dapat dianggap sebagai aturan yang paling kuat secara statistik dalam dataset. Namun, kekuatan aturan tetap perlu dilihat bersama nilai support dan confidence agar aturan tersebut tidak hanya kuat tetapi juga memiliki frekuensi dan tingkat kepercayaan yang memadai.

In [35]:
if len(rules) > 0:

    aturan_terkuat = rules.iloc[0]

    print("Aturan dengan Lift tertinggi:")
    print("Antecedents :", aturan_terkuat['antecedents'])
    print("Consequents :", aturan_terkuat['consequents'])
    print("Support     :", round(aturan_terkuat['support'], 3))
    print("Confidence  :", round(aturan_terkuat['confidence'], 3))
    print("Lift        :", round(aturan_terkuat['lift'], 3))

else:

    print("Tidak ada aturan yang memenuhi kriteria.")

Aturan dengan Lift tertinggi:
Antecedents : frozenset({np.str_('Keju'), np.str_('Teh')})
Consequents : frozenset({np.str_('Telur')})
Support     : 0.12
Confidence  : 0.857
Lift        : 2.381


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Interpretasi Aturan Asosiasi

Jika aturan Roti → Selai muncul sebagai salah satu aturan dengan nilai Lift yang tinggi, maka terdapat hubungan positif antara pembelian Roti dan Selai dalam dataset.

Secara bisnis, aturan tersebut masuk akal karena Roti dan Selai merupakan produk yang dapat digunakan secara bersamaan. Informasi seperti ini dapat dimanfaatkan untuk membuat paket promosi, rekomendasi produk, atau penempatan produk yang berdekatan.

Namun, keputusan bisnis sebaiknya tidak hanya berdasarkan Lift. Support dan Confidence juga perlu diperhatikan untuk memastikan bahwa aturan tersebut cukup sering terjadi dan memiliki tingkat kepercayaan yang baik.

## Langkah 5 — Content-Based Filtering

Content-Based Filtering memberikan rekomendasi berdasarkan karakteristik atau atribut produk.

Pada praktikum ini digunakan kategori produk sebagai fitur. Setiap kategori diubah menjadi representasi One-Hot Encoding, kemudian Cosine Similarity digunakan untuk menghitung tingkat kemiripan antarproduk.

Produk yang memiliki kategori sama akan memiliki tingkat kemiripan yang tinggi.

In [36]:
# Mengubah kategori menjadi fitur One-Hot

fitur = pd.get_dummies(
    katalog['kategori']
)

print("Fitur kategori:")

display(fitur)

# Menghitung Cosine Similarity

sim_matrix = cosine_similarity(
    fitur
)

print("Shape similarity matrix:", sim_matrix.shape)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Fitur kategori:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Bakery,Bumbu,Dairy,Minuman
0,True,False,False,False
1,True,False,False,False
2,False,False,True,False
3,True,False,False,False
4,False,False,True,False
5,False,False,True,False
6,False,False,False,True
7,False,True,False,False
8,False,False,False,True
9,False,False,True,False


Shape similarity matrix: (10, 10)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [37]:
def rekomendasi_serupa(
    nama_produk,
    top_n=3
):

    # Mencari index produk
    idx = katalog.index[
        katalog['produk'] == nama_produk
    ][0]

    # Mengambil skor similarity
    skor = list(
        enumerate(sim_matrix[idx])
    )

    # Mengurutkan dari similarity terbesar
    skor = sorted(
        skor,
        key=lambda x: x[1],
        reverse=True
    )

    # Menghapus produk itu sendiri
    skor = [
        s for s in skor
        if s[0] != idx
    ][:top_n]

    # Mengembalikan nama produk
    return katalog.iloc[
        [i for i, _ in skor]
    ]['produk'].tolist()


    print(
    "Mirip dengan Roti:",
    rekomendasi_serupa('Roti')
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [38]:
# Menampilkan similarity Roti terhadap semua produk

idx_roti = katalog.index[
    katalog['produk'] == 'Roti'
][0]

similarity_roti = pd.DataFrame({
    'produk': katalog['produk'],
    'kategori': katalog['kategori'],
    'similarity': sim_matrix[idx_roti]
})

similarity_roti = similarity_roti[
    similarity_roti['produk'] != 'Roti'
].sort_values(
    'similarity',
    ascending=False
)

display(similarity_roti)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,produk,kategori,similarity
1,Selai,Bakery,1.0
3,Sereal,Bakery,1.0
2,Susu,Dairy,0.0
4,Telur,Dairy,0.0
5,Keju,Dairy,0.0
6,Kopi,Minuman,0.0
7,Gula,Bumbu,0.0
8,Teh,Minuman,0.0
9,Mentega,Dairy,0.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Interpretasi Content-Based Filtering

Content-Based Filtering merekomendasikan produk berdasarkan kesamaan karakteristik. Karena fitur yang digunakan hanya kategori, produk yang berada pada kategori yang sama akan memiliki nilai Cosine Similarity yang tinggi.

Sebagai contoh, Roti berada dalam kategori Bakery sehingga produk lain dalam kategori Bakery seperti Selai dan Sereal memiliki tingkat kemiripan yang tinggi terhadap Roti. Pendekatan ini tidak menggunakan riwayat transaksi, tetapi hanya menggunakan karakteristik produk.

## Langkah 6 — Perbandingan Association Rules dan Content-Based Filtering

Pada tahap ini rekomendasi untuk produk Roti dibandingkan menggunakan dua pendekatan.

Association Rules menggunakan pola pembelian aktual dalam transaksi. Sementara itu, Content-Based Filtering menggunakan kemiripan atribut atau kategori produk.

Perbandingan ini digunakan untuk melihat apakah kedua pendekatan menghasilkan rekomendasi yang sama atau berbeda.

In [39]:
produk_target = 'Roti'

# Mencari aturan yang memiliki Roti sebagai antecedent

rules_terkait = rules[
    rules['antecedents'].apply(
        lambda x: produk_target in x
    )
]

print("Rekomendasi dari Association Rules:")

if len(rules_terkait) > 0:

    display(
        rules_terkait[
            [
                'antecedents',
                'consequents',
                'support',
                'confidence',
                'lift'
            ]
        ].head()
    )

else:

    print(
        "Tidak terdapat aturan dengan Roti sebagai antecedent."
    )

print("\nRekomendasi dari Content-Based Filtering:")

print(
    rekomendasi_serupa(
        produk_target
    )
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Rekomendasi dari Association Rules:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
12,"(Gula, Roti)",(Selai),0.10,1.0000,1.923077
1,(Roti),(Selai),0.22,0.6875,1.322115


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


Rekomendasi dari Content-Based Filtering:
['Selai', 'Sereal', 'Susu']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Interpretasi Perbandingan

Association Rules dan Content-Based Filtering memiliki cara kerja yang berbeda.

Association Rules menghasilkan rekomendasi berdasarkan pola pembelian pelanggan. Jika pelanggan yang membeli Roti sering membeli produk tertentu, produk tersebut dapat direkomendasikan berdasarkan aturan asosiasi.

Content-Based Filtering menghasilkan rekomendasi berdasarkan kemiripan karakteristik produk. Pada praktikum ini karakteristik yang digunakan adalah kategori produk.

Kedua pendekatan dapat menghasilkan rekomendasi yang sama maupun berbeda. Perbedaan tersebut terjadi karena Association Rules menggunakan perilaku pembelian, sedangkan Content-Based Filtering menggunakan atribut produk.

## Penggunaan Setiap Pendekatan

Association Rules cocok digunakan ketika perusahaan memiliki data transaksi yang cukup banyak dan ingin mengetahui pola pembelian produk yang sering terjadi secara bersamaan.

Content-Based Filtering cocok digunakan ketika informasi mengenai karakteristik produk tersedia, tetapi data interaksi atau transaksi pelanggan masih terbatas.

Kedua pendekatan juga dapat digabungkan menjadi sistem rekomendasi hybrid. Association Rules dapat memberikan rekomendasi berdasarkan perilaku pembelian, sedangkan Content-Based Filtering dapat memberikan rekomendasi berdasarkan kemiripan produk. Dengan menggabungkan keduanya, sistem dapat menghasilkan rekomendasi yang lebih beragam.

## Ringkasan Hasil Praktikum

Pada praktikum ini telah dilakukan analisis transaksi menggunakan algoritma Apriori dan pembuatan sistem rekomendasi menggunakan Content-Based Filtering.

Dataset terdiri dari 50 transaksi dan 10 jenis produk. Data transaksi diubah menjadi One-Hot Encoding sebelum digunakan dalam algoritma Apriori. Frequent itemset kemudian ditemukan menggunakan beberapa nilai min_support dan aturan asosiasi dibentuk berdasarkan confidence serta disaring menggunakan nilai lift.

Content-Based Filtering dibuat menggunakan kategori produk sebagai fitur. Cosine Similarity digunakan untuk menentukan produk yang memiliki karakteristik paling mirip dengan produk target.

Hasil kedua pendekatan kemudian dibandingkan. Association Rules berfokus pada pola pembelian pelanggan, sedangkan Content-Based Filtering berfokus pada kemiripan karakteristik produk.

# Kesimpulan

Pada praktikum ini telah dipelajari penerapan Association Rule Mining menggunakan algoritma Apriori dan sistem rekomendasi sederhana menggunakan Content-Based Filtering. Algoritma Apriori digunakan untuk menemukan pola produk yang sering dibeli secara bersamaan berdasarkan Support, Confidence, dan Lift. Sementara itu, Content-Based Filtering menggunakan kategori produk dan Cosine Similarity untuk menemukan produk yang memiliki karakteristik serupa. Kedua metode memiliki kelebihan yang berbeda, karena Association Rules memanfaatkan pola transaksi pelanggan sedangkan Content-Based Filtering memanfaatkan atribut produk. Keterbatasan praktikum ini adalah dataset yang digunakan masih berupa data sintetis dan jumlah transaksi relatif kecil, sehingga aturan yang ditemukan belum tentu menggambarkan pola pembelian pada kondisi nyata.